# 1.5.2. Perkėlimo mokymasis ir papildomas mokymas gėlėms

**Liutauras Turauskis · DIPf-24 · ND1 tęsinys**

Pritaikyta pagal dėstytojo pateiktą `cnn_Transfer_learning.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. CIFAR-10 pakeistas gėlių rinkiniu. Išlaikyti užšaldytas požymių išskyrimas, naujos galvos mokymas, išsaugojimas, įkėlimas, duomenų augmentavimas ir fine-tuning.

**Paleidimas:** Google Colab → „Runtime“ → „Run all“. Ląsteles vykdyti nuo pradžios.
Duomenys atsisiunčiami automatiškai; savo kodo, failų ir API rakto įterpti nereikia.
GPU rekomenduojamas vaizdų mokymui, bet nėra būtinas. Pirmas atsisiuntimas gali užtrukti.

**Rezultatų sąžiningumas:** šioje pateikiamoje kopijoje mokymo išvestys išvalytos.
Rodikliai, grafikai ir konkrečios išvados apskaičiuojami paleidus, o ne įrašomi iš anksto.
Parengimo aplinkoje TensorFlow mokymas nebuvo vykdytas; prieš pateikiant reikia paleisti visą užrašyną
Colab ir išsaugoti kopiją su išvestimis. Atsitiktinė sėkla mažina rezultatų svyravimą, tačiau GPU
ir bibliotekų versijos gali lemti nedidelius skirtumus.


In [ ]:
%pip -q install "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" requests pillow


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report)
from IPython.display import display
SEED = 42
OUT = Path('rezultatai')
OUT.mkdir(exist_ok=True)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


## 1. Pasirinkti vaizdų duomenys

Originalo CIFAR-10 / interneto nuotraukos pakeistos **Flowers Photos** rinkiniu:
`daisy`, `dandelion`, `roses`, `sunflowers`, `tulips` (5 klasės).
[TensorFlow šaltinis](https://www.tensorflow.org/tutorials/load_data/images).
Visas archyvas užima apie 218 MB. Nuotraukų autoriai ir CC BY licencijos pateikti archyvo `LICENSE.txt`.
Tai vieši pasirinkti duomenys, o ne mano paties fotografuotos nuotraukos.

Eksperimentui atsitiktinai pasirenkama iki 200 nuotraukų iš kiekvienos klasės.
Pasikartojantys vienodi failų turiniai atmetami prieš skaidymą. Mokymo, validavimo ir testo dalys:
70 %, 15 %, 15 %. Visuose trijuose vaizdų užrašynuose parinkimo taisyklės vienodos.
Panašių, bet ne identiškų nuotraukų šis patikrinimas neaptinka.


In [ ]:
from PIL import Image, ImageDraw
from collections import Counter
import tarfile
CACHE = Path('/content/nd1_duomenys') if Path('/content').exists() else Path('nd1_duomenys')
CACHE.mkdir(exist_ok=True)
archive = keras.utils.get_file(fname='flower_photos.tgz',
    origin='https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
    cache_dir=str(CACHE.resolve()), cache_subdir='archyvai', extract=False)
DATA = CACHE / 'flower_photos'
if not DATA.exists():
    with tarfile.open(archive) as tar:
        tar.extractall(CACHE, filter='data')
classes = sorted(p.name for p in DATA.iterdir() if p.is_dir())
assert classes == ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
rng = np.random.default_rng(SEED)
rows, seen = [], set()
for label, name in enumerate(classes):
    files = sorted((DATA / name).glob('*.jpg'))
    # Atsitiktinimas atliekamas po rūšiavimo, kad failų sistemos tvarka nekeistų imties.
    files = [files[int(i)] for i in rng.permutation(len(files))]
    selected = 0
    for path in files:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        rows.append({'path': str(path), 'relative_path': str(path.relative_to(DATA)),
                     'label': label, 'class_name': name, 'sha256': digest})
        selected += 1
        if selected == 200:
            break
images = pd.DataFrame(rows)
train_img, remaining = train_test_split(images, test_size=.30, stratify=images.label, random_state=SEED)
val_img, test_img = train_test_split(remaining, test_size=.50, stratify=remaining.label, random_state=SEED)
assert set(train_img.sha256).isdisjoint(val_img.sha256)
assert set(train_img.sha256).isdisjoint(test_img.sha256)
assert set(val_img.sha256).isdisjoint(test_img.sha256)
manifest = pd.concat([part.assign(split=name) for name, part in
    [('train', train_img), ('val', val_img), ('test', test_img)]])
manifest.drop(columns='path').to_csv(OUT / 'geliu_padalinimas.csv', index=False)
display(pd.crosstab(manifest.class_name, manifest.split))
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, name in zip(axes, classes):
    path = train_img.loc[train_img.class_name == name, 'path'].iloc[0]
    ax.imshow(Image.open(path).convert('RGB')); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()


## 2. Įvestis ir duomenų augmentavimas

Originalo TF Hub EfficientNet sluoksnis pakeistas Keras MobileNetV2: modelis lengvesnis ir
nesukelia Keras 2 / Keras 3 maišymo. Naudojami ImageNet svoriai, tačiau 1000 klasių galva pašalinama.
Vaizdai yra 160×160 RGB. Apvertimas, pasukimas ir priartinimas atliekami tik mokymo metu;
validavimo ir testo vaizdai neaugmentuojami. Nauja galva turi 5 gėlių klases.


In [ ]:
SIZE = 96  # Vienoda įvestis CNN ir MLP palyginimui.
BATCH = 32
def image_dataset(frame, training=False, size=SIZE):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(str), frame.label.to_numpy('int32')))
    def decode(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        # Ši funkcija palieka reikšmes 0..255; reikiamas mastelis nustatomas modelyje.
        return tf.image.resize(tf.cast(image, tf.float32), [size, size]), label
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = image_dataset(train_img, True), image_dataset(val_img), image_dataset(test_img)


In [ ]:
SIZE = 160
train_ds = image_dataset(train_img, True, size=SIZE)
val_ds = image_dataset(val_img, size=SIZE)
test_ds = image_dataset(test_img, size=SIZE)
augmentation = keras.Sequential([layers.RandomFlip('horizontal', seed=SEED),
    layers.RandomRotation(.08, seed=SEED+1), layers.RandomZoom(.10, seed=SEED+2)], name='augmentavimas')
batch, _ = next(iter(train_ds))
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for ax in axes:
    ax.imshow(np.clip(augmentation(batch[:1], training=True).numpy()[0] / 255.0, 0, 1)); ax.axis('off')
plt.suptitle('Vienos mokymo nuotraukos variantai'); plt.tight_layout(); plt.show()
base = keras.applications.MobileNetV2(input_shape=(SIZE, SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False
inputs = keras.Input(shape=(SIZE, SIZE, 3))
x = augmentation(inputs)
x = layers.Rescaling(1./127.5, offset=-1)(x)  # MobileNetV2 įvestis: -1..1.
x = base(x, training=False)  # BatchNormalization statistika išsaugoma ir fine-tuning metu.
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(.2)(x)
outputs = layers.Dense(len(classes), activation='softmax')(x)
model = keras.Model(inputs, outputs, name='flowers_transfer')
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
print('Mokomų parametrų:', sum(int(np.prod(v.shape)) for v in model.trainable_weights))


## 3. Naujos galvos mokymas ir išsaugojimas

Užšaldytas bazinis tinklas išskiria bendrus vaizdo požymius; mokomi tik naujos galvos svoriai.
Validavimas naudojamas ankstyvam stabdymui ir geriausios epochos parinkimui.
Originale modelis buvo saugomas ir įkeliamas; čia naudojamas vientisas `.keras` formatas.
Iškart patikrinama, ar įkėlus prognozės sutampa.


In [ ]:
history_head = model.fit(train_ds, validation_data=val_ds, epochs=6,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=2)
model.save(OUT / 'flowers_frozen.keras')
frozen = keras.models.load_model(OUT / 'flowers_frozen.keras')
sample, _ = next(iter(val_ds))
np.testing.assert_allclose(model(sample, training=False).numpy(), frozen(sample, training=False).numpy(), atol=1e-5)
frozen_val_loss = float(frozen.evaluate(val_ds, verbose=0)[0])
print('Užšaldytas modelis įkeltas; prognozės sutapo. Validavimo loss:', frozen_val_loss)


In [ ]:
def image_metrics(name, model):
    # Testo duomenų tvarka nekeičiama, todėl sutampa su test_img etiketėmis.
    probability = model.predict(test_ds, verbose=0)
    predicted = probability.argmax(axis=1)
    y = test_img.label.to_numpy()
    result = {'modelis': name, 'accuracy': accuracy_score(y, predicted),
              'macro_f1': f1_score(y, predicted, average='macro', zero_division=0)}
    print(classification_report(y, predicted, labels=range(len(classes)), target_names=classes, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y, predicted, labels=range(len(classes)),
        display_labels=classes, xticks_rotation=35, cmap='Blues')
    plt.title(name + ': nepriklausomas testas'); plt.tight_layout(); plt.show()
    return result

def show_history(history, title):
    frame = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, metric in zip(axes, ['loss', 'accuracy']):
        ax.plot(frame[metric], label='mokymas')
        ax.plot(frame['val_' + metric], label='validavimas')
        ax.set(xlabel='Epocha (nuo 0)', title=title + ': ' + metric); ax.legend()
    plt.tight_layout(); plt.show()


In [ ]:
show_history(history_head, 'Užšaldyta bazė')


## 4. Fine-tuning: dalies bazės papildomas mokymas

Atšaldomi paskutiniai 20 bazinio tinklo sluoksnių, išskyrus `BatchNormalization`.
Pakeitus `trainable`, būtina **iš naujo compile**. Mokymosi žingsnis sumažinamas nuo 0,001 iki 0,00001,
kad neišardytume išmoktų požymių. Originalo tiesioginio `optimizer.lr` keitimo čia nereikia.
Tai atskira antroji mokymo fazė; ankstyvas stabdymas ir vėl naudoja tik validavimą.


In [ ]:
base.trainable = True
for layer in base.layers[:-20]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Mokomų parametrų po atšaldymo:', sum(int(np.prod(v.shape)) for v in model.trainable_weights))
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=5,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)], verbose=2)
show_history(history_fine, 'Fine-tuning')
fine_val_loss = float(model.evaluate(val_ds, verbose=0)[0])
selected = model if fine_val_loss < frozen_val_loss else frozen
selected_phase = 'fine_tuning' if fine_val_loss < frozen_val_loss else 'uzsaldyta_baze'
print('Modelis parinktas pagal validavimo loss:', selected_phase)
selected.save(OUT / 'flowers_galutinis.keras')
pd.DataFrame(history_head.history).to_csv(OUT / 'transfer_galvos_eiga.csv', index=False)
pd.DataFrame(history_fine.history).to_csv(OUT / 'transfer_fine_eiga.csv', index=False)


## 5. Galutinis nepriklausomas testas

Abi fazės vertinamos tame pačiame teste, tačiau galutinis modelis jau pasirinktas pagal validavimą.
Jei fine-tuning nepagerina validavimo nuostolio, lieka užšaldytas variantas – pagerėjimas nėra garantuotas.
Testo rezultatai nekeičiami į modelio parinkimo taisyklę. Rodoma macro F1, accuracy, painiavos matricos ir prognozių pavyzdžiai.


In [ ]:
results = pd.DataFrame([image_metrics('Užšaldyta bazė', frozen), image_metrics('Fine-tuning', model)])
display(results)
results.to_csv(OUT / 'transfer_testas.csv', index=False)
final = keras.models.load_model(OUT / 'flowers_galutinis.keras')
np.testing.assert_allclose(selected(sample, training=False).numpy(), final(sample, training=False).numpy(), atol=1e-5)
test_batch, test_labels = next(iter(test_ds))
probabilities = final(test_batch[:8], training=False).numpy()
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    prediction = int(probabilities[i].argmax())
    truth = int(test_labels[i])
    ax.imshow(np.clip(test_batch[i].numpy() / 255.0, 0, 1))
    ax.set_title(f'Tikra: {classes[truth]}\nSpėta: {classes[prediction]} ({probabilities[i].max():.2f})', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()
(OUT / 'geliu_klases.json').write_text(json.dumps(classes, ensure_ascii=False), encoding='utf-8')
(OUT / 'pasirinkta_faze.json').write_text(json.dumps({'phase': selected_phase,
    'frozen_val_loss': frozen_val_loss, 'fine_val_loss': fine_val_loss}, indent=2))
print('Galutinis modelis:', selected_phase, '| Išsaugojimas ir įkėlimas patikrinti.')


## 6. Kuo skiriasi abu 1.5 failai?

1.5.1 panaudojo gatavus modelius, kurių svoriai nebuvo keičiami. Šiame 1.5.2 užrašyne
ImageNet požymiai perkeliami į naują penkių gėlių klasių užduotį: pirma mokoma galva,
paskui maža bazės dalis. Augmentavimas didina mokymo pavyzdžių įvairovę, bet nesukuria
naujų nepriklausomų testo duomenų. Rezultatai galioja šio rinkinio imčiai; kitokios aplinkos
nuotraukos ar kelios gėlės viename kadre gali duoti prastesnes prognozes.

Šaltiniai: [TensorFlow transfer learning ir fine-tuning](https://www.tensorflow.org/tutorials/images/transfer_learning),
[Keras MobileNetV2](https://keras.io/api/applications/mobilenet/).
